In [1]:
import pandas as pd

In [12]:
weather = pd.read_csv('../data/raw/nasa_monthly_weather_2014_2025.csv')
df_final = pd.read_csv('../data/processed/clean_crop_yield_data.csv')


In [ ]:
# Extract Year and Month

weather["Year"] = weather["YearMonth"] // 100
weather["Month"] = weather["YearMonth"] % 100

print(weather.head())

                         State                  District  YearMonth  \
0  Andaman And Nicobar Islands  North And Middle Andaman     201401   
1  Andaman And Nicobar Islands  North And Middle Andaman     201402   
2  Andaman And Nicobar Islands  North And Middle Andaman     201403   
3  Andaman And Nicobar Islands  North And Middle Andaman     201404   
4  Andaman And Nicobar Islands  North And Middle Andaman     201405   

   Rainfall_mm  Temperature_C  Year  Month  
0         0.66          26.14  2014      1  
1         0.01          26.26  2014      2  
2         0.02          28.18  2014      3  
3         0.36          30.24  2014      4  
4         9.14          29.29  2014      5  


In [10]:
# Aggregate Monthly → Yearly

weather_yearly = (
    weather
    .groupby(["State", "District", "Year"])
    .agg({
        "Rainfall_mm": "sum",      # total yearly rainfall
        "Temperature_C": "mean"    # average yearly temperature
    })
    .reset_index()
)

print("Weather yearly shape:", weather_yearly.shape)

Weather yearly shape: (8544, 5)


In [14]:
# Merge With Yield Data

df_merged = df_final.merge(
    weather_yearly,
    on=["State", "District", "Year"],
    how="left"
)

print("After merge:", df_merged.shape)
print(df_merged.isnull().sum())

After merge: (44324, 9)
State              0
District           0
Crop               0
Season             0
Area               0
Yield              0
Year               0
Rainfall_mm      961
Temperature_C    961
dtype: int64


In [15]:
# Check current shape
print("Before dropping null climate rows:", df_merged.shape)

# Drop rows where rainfall or temperature is missing
df_merged = df_merged.dropna(
    subset=["Rainfall_mm", "Temperature_C"]
).copy()

print("After dropping null climate rows:", df_merged.shape)

# Verify no nulls remain
print(df_merged[["Rainfall_mm", "Temperature_C"]].isnull().sum())

Before dropping null climate rows: (44324, 9)
After dropping null climate rows: (43363, 9)
Rainfall_mm      0
Temperature_C    0
dtype: int64


In [16]:
# Ensure numeric types are correct
df_merged["Rainfall_mm"] = pd.to_numeric(df_merged["Rainfall_mm"], errors="coerce")
df_merged["Temperature_C"] = pd.to_numeric(df_merged["Temperature_C"], errors="coerce")

# Final null check
print(df_merged.isnull().sum())

State            0
District         0
Crop             0
Season           0
Area             0
Yield            0
Year             0
Rainfall_mm      0
Temperature_C    0
dtype: int64


In [ ]:
# Save cleaned dataset
df_merged.to_csv(
    "../data/processed/clean_yield_with_climate.csv",
    index=False
)

print("Saved successfully ✅")

Saved successfully ✅
